In [ ]:
OPENROUTER_API_KEY="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"

In [12]:
# core/llm_parser.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

from src.custom_exception import CustomException
from src.logger import get_logger

logger = get_logger("llm_parser")

async def extract_preferences(mensaje: str, model: str, api_key: str) -> dict:
    try:
        logger.info("##### INITIALIZING LLM PARSER.PY ##### ")

        try:
                logger.info("##### Creating RAG chain #####")
                prompt = """
                            You are an assistant that analyzes cultural preferences.\n"
                                "From the user's text, extract a list of:\n"
                                "- 'likes': things they like (e.g., 'Radiohead', 'existentialist novels')\n"
                                "- 'dislikes': things they dislike (optional)\n"
                                "- 'categories': related content types (e.g., music, books, travel, movies...)\n\n"
                                "Return recommendations in strict JSON format only."
                                This is the user's text: {document}
                        """
                llm = ChatOpenAI(
                    base_url="https://openrouter.ai/api/v1",
                    openai_api_key=api_key,
                    model=model
                )
                prompt_template = ChatPromptTemplate.from_template(prompt)
                logger.info("##### FINISHED - Creating RAG chain #####")
        except Exception as e:
                logger.error(f"Error while Creating RAG chain: {e}")
                raise CustomException("Error while Creating RAG chain", e)
        try:
                logger.info("##### GETTING THE ANSWER FROM CHAIN #####")
                answer = prompt_template | llm | StrOutputParser()
                final_answer = answer.invoke({"document": mensaje})
                logger.info("##### FINISHED - GETTING THE ANSWER FROM CHAIN #####")
        except Exception as e:
                logger.error(f"Error while GETTING THE ANSWER FROM CHAIN: {e}")
                raise CustomException("Error GETTING THE ANSWER FROM CHAIN", e)
        
        logger.info("##### FINISHED -- INITIALIZING LLM PARSER.PY ##### ")
        return final_answer
    except Exception as e:
        logger.error(f"ERROR INITIALIZING LLM PARSER.PY: {e}")
        raise CustomException("ERROR INITIALIZING LLM PARSER.PY", e)




In [21]:
parsed_response = await extract_preferences("I like science fiction movies and play video games", "deepseek/deepseek-chat-v3-0324:free", OPENROUTER_API_KEY)

In [22]:
parsed_response

'```json\n{\n  "likes": ["science fiction movies", "video games"],\n  "dislikes": [],\n  "categories": ["movies", "video games"]\n}\n```'

In [ ]:
import json

# 1. Eliminar etiquetas de bloque Markdown
cleaned = parsed_response.strip("` \n")

# 2. Si aún contiene la palabra "json", quitarla
if cleaned.startswith("json"):
    cleaned = cleaned[len("json"):].strip()

# 3. Cargar el JSON
try:
    parsed = json.loads(cleaned)
    print(parsed["likes"])        # ['science fiction movies', 'video games']
    print(parsed["categories"])   # ['movies', 'video games']
except json.JSONDecodeError:
    print("❌ Error al decodificar el JSON")


['science fiction movies', 'video games']
['movies', 'video games']


In [26]:
import requests
import pandas as pd
import time

# Qloo Hackathon API details
API_KEY = "nPdmCgeNok-stq190jXiQUMF0sGZuJYSs0LVMyAmUo0"
BASE_URL = "https://hackathon.api.qloo.com"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

def get_recommendations(movie_name):
    """Fetch recommendations for a given movie name"""
    try:
        # Step 1: Get entity tags
        tags_url = f"{BASE_URL}/tags"
        tags_payload = {"name": movie_name, "type": "movie", "take": 1}

        tags_response = requests.post(tags_url, json=tags_payload, headers=headers)
        if tags_response.status_code != 200:
            return []
        
        tags_data = tags_response.json()
        if not tags_data.get("results"):
            return []

        entity_id = tags_data["results"][0]["id"]

        # Step 2: Get recommendations
        insights_url = f"{BASE_URL}/insights"
        insights_payload = {
            "filterType": "urn:entity:movie",
            "signalInterestEntities": [entity_id],
            "take": 5
        }

        insights_response = requests.post(insights_url, json=insights_payload, headers=headers)
        if insights_response.status_code != 200:
            return []
        
        recommendations_data = insights_response.json()
        recs = recommendations_data.get("results", {}).get("recommendations", [])
        return [rec["name"] for rec in recs]
    
    except Exception as e:
        print(f"⚠️ Error for {movie_name}: {e}")
        return []

if __name__ == "__main__":
    # Load your cleaned dataset (update file name if needed)
    input_file = "cleaned_dataset.csv"
    output_file = "recommendations_output.csv"

    df = pd.read_csv(input_file)

    # Assuming your CSV has a 'movie_name' column
    if "name" not in df.columns:
        print("❌ CSV must have a column named 'name'")
        exit()

    results = []

    for movie in df["name"]:
        print(f"🎬 Fetching recommendations for: {movie}")
        recs = get_recommendations(movie)
        results.append({"movie_name": movie, "recommendations": "; ".join(recs)})
        time.sleep(0.5)  # avoid hitting API too fast

    output_df = pd.DataFrame(results)
    output_df.to_csv(output_file, index=False)
    print(f"\n✅ Recommendations saved to {output_file}")


FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_dataset.csv'